**MMA 662: First Quiz**

## Instructions:

1. This quiz has 130 points, so it has 30 bonus points, but the grade cannot be more than 100: $\text{grade} = \min\{\text{points earned}, 100\}$
2. Remember the independency rule we had in the assignment. The code for different questions must run independently of each other.
3. The quiz begins at 8:30 and finishes at 9:55 AM. Submissions after 9:55 AM are not accepted.
4. You must submit one Jupyter notebook that contains your code and the results. The grader should be able to reproduce the results you reported by running your code. A complete, working code is part of the answer.
5. Don't forget that Python indexing system starts from 0 and not 1.

In [2]:
# Run this cell first!
import gurobipy as gp
from gurobipy import *

import numpy as np

## The story:

The salesforce of a firm consists of ten salespeople. Because they had a record-breaking season, Ms. Fraser - the manager - decided to award \\$10,000 to the sales team. A wise manager, Ms. Fraser visits the head of the HR team, Mr. Lei, before making any decisions. She wants to know the best way to distribute the award among the salespeople. Mr. Lei, who is also clever, understands that she has something in mind, so he joyfully replies: "It depends! What do you mean by *the best*?" 

In this quiz, we accompany them in their decision making journey to discuss different scenarios.

## Question 1:

Mr. Lei explains that there is a psychological payoff due to the **spillover effect** among the salespeople. That is, the friendship among them affects their feelings, so they become happier if their friends receive some awards in addition to what they receive. Therefore, the total payoff of each salesperson consists of two parts: his or her own monetary payoff (the money he or she receives) and the psychological payoff (the effect of happiness of their friends due to the money they receive.) 

**From now on, we will call the total payoff simply as the payoff**.

Mr. Lei then draws the following matrix (called **A**) to show among whom these effects exist and what their intensity is:


| | $S_1$ | $S_2$ | $S_3$ | $S_4$ | $S_5$ | $S_6$ | $S_7$ | $S_8$ | $S_9$ | $S_{10}$|
| -- | -- | -- | -- | -- | -- | -- | -- | -- | -- | -- |
|$S_1$|1 | 0.3 | 0.2 | 0 | 0  | 0  | 0.2  | 0 | 0.4 | 0 |
|$S_2$ | 0.2 | 1 | 0.2 | 0 | 0.1 | 0 | 0.2 | 0.1 | 0.3 | 0 |
|$S_3$| 0.2 | 0.1 | 1 | 0.1 | 0 | 0 | 0.4 | 0.2 | 0 | 0 |
|$S_4$| 0 | 0.1 | 0.2 | 1 | 0.3 | 0 | 0.1 | 0.2 | 0.3 | 0.1 |
|$S_5$| 0.1 | 0.1 | 0 | 0.2 | 1 | 0.1 | 0.2 | 0 | 0 | 0.1 | 
|$S_6$| 0 | 0 | 0 | 0.1 | 0.1 | 1 | 0.1 | 0 | 0 | 0 |
|$S_7$| 0.2 | 0.2 | 0 | 0.1 | 0.3 | 0 | 1 | 0.1 | 0.2 | 0.3 |
|$S_8$| 0 | 0 | 0.3 | 0 | 0.1 | 0 | 0.3 | 1 | 0.1 | 0.3 |
|$S_9$| 0.4 | 0.4 | 0 | 0.1 | 0 | 0 | 0.1 | 0.2 | 1 | 0.1 |
|$S_{10}$| 0 | 0 | 0.1 | 0.2 | 0.1 | 0 | 0.3 | 0.3 | 0.2 | 1 |

where $a_{i,j}$ (the element in the intersection of the i'th row and the j'th column) is the payoff salesperson $i$ feels when salesperson $j$ receives one dollar. The payoff is linear in the received award. This means: $$\text{payoff}_i = \sum_{j = 1}^{10} a_{i,j} x_j$$
where $x_j$ is the amount of money salesperson $j$ receives in the allocation. Suppose, for instance, Ms. Fraser gives \\$2,000 to salesperson 1, \\$3,000 to salesperson 3, \\$3,000 to salesperson 5, and \\$2,000 to salesperson 7, the payoff of salesperson 3 would be:
$$\text{payoff}_3 = \sum_{j = 1}^{10} a_{3,j} x_j = 0.2 \times 2000 + 1 \times 3000 + 0 \times 3000 + 0.4 \times 2000 = 4200$$.

The following example will further help you understand in more details what this model means. Suppose Ms. Fraser gives \\$5,000 to salesperson 1 and \\$5,000 to salesperson 2. The following code will calculate the payoff of each salesperson resulting from this allocation:

In [3]:


A = np.array([[1, 0.3, 0.2, 0, 0, 0, 0.2, 0, 0.4, 0],
              [0.2, 1, 0.2, 0, 0.1, 0, 0.2, 0.1, 0.3, 0],
              [0.2, 0.1, 1, 0.1, 0, 0, 0.4, 0.2, 0, 0],
              [0, 0.1, 0.2, 1, 0.3, 0, 0.1, 0.2, 0.3, 0.1],
              [0.1, 0.1, 0, 0.2, 1, 0.1, 0.2, 0, 0, 0.1],
              [0, 0, 0, 0.1, 0.1, 1, 0.1, 0, 0, 0],
              [0.2, 0.2, 0, 0.1, 0.3, 0, 1, 0.1, 0.2, 0.3],
              [0, 0, 0.3, 0, 0.1, 0, 0.3, 1, 0.1, 0.3],
              [0.4, 0.4, 0, 0.1, 0, 0, 0.1, 0.2, 1, 0.1],
              [0, 0, 0.1, 0.2, 0.1, 0, 0.3, 0.3, 0.2, 1]])

allocation = np.array([5000, 5000, 0, 0, 0, 0, 0, 0, 0, 0])
payoff_A = np.empty(10)

# Let's calculate the payoff of salesperson i from this allocation:
for i in range(10): 
    payoff_A[i] = sum(A[i][j] * allocation[j] for j in range(10))
    
for i in range(10):
    print("The payoff of salesperson", (i+1), "from this allocation is %d" % payoff_A[i])
    
# The reason for adding 1 to the index is that Python indices start from 0, while our salesperson indices start from 1.
# Therefore, for instance, payoff_A[6] correspondes to the payoff of salesperson 7, not 6.

The payoff of salesperson 1 from this allocation is 6500
The payoff of salesperson 2 from this allocation is 6000
The payoff of salesperson 3 from this allocation is 1500
The payoff of salesperson 4 from this allocation is 500
The payoff of salesperson 5 from this allocation is 1000
The payoff of salesperson 6 from this allocation is 0
The payoff of salesperson 7 from this allocation is 2000
The payoff of salesperson 8 from this allocation is 0
The payoff of salesperson 9 from this allocation is 4000
The payoff of salesperson 10 from this allocation is 0


And the **average payoff** (the sum of the payoff of all salespersons divided by their number) with this allocation would be:

In [3]:
average_payoff = sum(payoff_A[i] for i in range(10)) / 10
print("The average payoff of this allocation is %d." % average_payoff)

The average payoff of this allocation is 2150.


### Now that you understood what is going on here, consider the following cases and answer the questions using matrix **A**:

### 1.1: (10 pts)
Suppose Ms. Fraser deeply believes in equality, and her interpretation of equality is equally distributing the award among all ten salespeople - i.e., she allocates \\$1,000 to each. What is the average payoff of this policy?

In [1]:
# Your solution for 1.1
# Run this cell first!
import gurobipy as gp
from gurobipy import *
import numpy as np

A_1_1 = np.array([[1, 0.3, 0.2, 0, 0, 0, 0.2, 0, 0.4, 0],
              [0.2, 1, 0.2, 0, 0.1, 0, 0.2, 0.1, 0.3, 0],
              [0.2, 0.1, 1, 0.1, 0, 0, 0.4, 0.2, 0, 0],
              [0, 0.1, 0.2, 1, 0.3, 0, 0.1, 0.2, 0.3, 0.1],
              [0.1, 0.1, 0, 0.2, 1, 0.1, 0.2, 0, 0, 0.1],
              [0, 0, 0, 0.1, 0.1, 1, 0.1, 0, 0, 0],
              [0.2, 0.2, 0, 0.1, 0.3, 0, 1, 0.1, 0.2, 0.3],
              [0, 0, 0.3, 0, 0.1, 0, 0.3, 1, 0.1, 0.3],
              [0.4, 0.4, 0, 0.1, 0, 0, 0.1, 0.2, 1, 0.1],
              [0, 0, 0.1, 0.2, 0.1, 0, 0.3, 0.3, 0.2, 1]])

allocation_1_1 = np.array([1000, 1000, 1000, 1000, 1000, 1000,1000 , 1000, 1000, 1000])
payoff_A_1_1 = np.empty(10)

# Let's calculate the payoff of salesperson i from this allocation:
for i in range(10): 
    payoff_A_1_1[i] = sum(A_1_1[i][j] * allocation_1_1[j] for j in range(10))
average_payoff_1_1 = sum(payoff_A_1_1[i] for i in range(10)) / 10
print("The average payoff of this allocation is %d." % average_payoff_1_1)

The average payoff of this allocation is 2060.


### 1.2: (15 pts)
Suppose Ms. Fraser now wants to maximize the average payoff. What is the optimal award allocation? What is the average payoff? (Hint: you can solve this without help from Gurobi or any optimization program. But if you are not 100% sure of your solution, I suggest playing safe and using Gurobi.)

In [2]:
# Your solution for 1.2

#There are likely multiple solutions for this program
#Since the team get satsifaction by having their bonus paid and others we will need to maximize the number given to members who have higher satsifaction from each other
#We simply need to maximize the total payoff 
#since there is no limit we can split the payoff between people who offer the highestspillover effect

import gurobipy as gp
from gurobipy import *
import numpy as np

A_1_2 = np.array([[1, 0.3, 0.2, 0, 0, 0, 0.2, 0, 0.4, 0],
              [0.2, 1, 0.2, 0, 0.1, 0, 0.2, 0.1, 0.3, 0],
              [0.2, 0.1, 1, 0.1, 0, 0, 0.4, 0.2, 0, 0],
              [0, 0.1, 0.2, 1, 0.3, 0, 0.1, 0.2, 0.3, 0.1],
              [0.1, 0.1, 0, 0.2, 1, 0.1, 0.2, 0, 0, 0.1],
              [0, 0, 0, 0.1, 0.1, 1, 0.1, 0, 0, 0],
              [0.2, 0.2, 0, 0.1, 0.3, 0, 1, 0.1, 0.2, 0.3],
              [0, 0, 0.3, 0, 0.1, 0, 0.3, 1, 0.1, 0.3],
              [0.4, 0.4, 0, 0.1, 0, 0, 0.1, 0.2, 1, 0.1],
              [0, 0, 0.1, 0.2, 0.1, 0, 0.3, 0.3, 0.2, 1]])
names_1_2=['S_1', 'S_2', 'S_3', 'S_4', 'S_5', 'S_6', 'S_7', 'S_8','S_9','S_10',]
People_1_2=len(A_1_2)

prob_1_2=gp.Model("Question 1.2")

#Create variable
#X1_2=prob_1_2.addVars(People_1_2,People_1_2,lb=0,vtype=GRB.CONTINUOUS,name = [i+"_to_"+j for i in names_1_2 for j in names_1_2])
X1_2=prob_1_2.addVars(People_1_2,lb=0,vtype=GRB.CONTINUOUS,name = [i for i in names_1_2])

#set objective function 
#need the spillover effect
#prob_1_2.setObjective(sum(X1_2[i,j]*A_1_2[i][j] for i in range(People_1_2)for j in range (People_1_2)),GRB.MAXIMIZE)

#trying new objective function
#The objective function seems to be placing all the amounts into one variable
#prob_1_2.setObjective(sum(A_1_2[i][j]*X1_2[i,j] for i in range(10) for j in range(10)),GRB.MAXIMIZE)
prob_1_2.setObjective(sum(A_1_2[i][j]*X1_2[i] for i in range(10) for j in range(10)),GRB.MAXIMIZE)

#set constraints
##nonnegativity
for i in range(People_1_2):
        #prob_1_2.addConstr(X1_2[i,j]>=0)
        prob_1_2.addConstr(X1_2[i]>=0)
    
#prob_1_2.addConstr(sum(X1_2[i,j] for i in range(People_1_2)for j in range(People_1_2))<=10000)
prob_1_2.addConstr(sum(X1_2[i] for i in range(People_1_2))<=10000)

#optimize problem
prob_1_2.optimize()    
    
print("The optimal allocation is as follows:")
for v in prob_1_2.getVars():
    print(v.varName, "=", round(v.x))
print("The average payoff is",round(prob_1_2.objVal)/10)

Set parameter Username
Academic license - for non-commercial use only - expires 2023-09-13
Gurobi Optimizer version 9.5.2 build v9.5.2rc0 (win64)
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 11 rows, 10 columns and 20 nonzeros
Model fingerprint: 0x83d94a00
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 2e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+04, 1e+04]
Presolve removed 11 rows and 10 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.4000000e+04   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  2.400000000e+04
The optimal allocation is as follows:
S_1 = 0
S_2 = 0
S_3 = 0
S_4 = 0
S_5 = 0
S_6 = 0
S_7 = 10000
S_8 = 0
S_9 = 0
S_10 = 0
The average payoff is 2400.0


### 1.3: (20 pts)
Suppose Ms. Fraser still wants to maximize the average payoff, but this time she wants to ensure that each salesperson will receive at least **1500 units of payoff**. (Note that to receive 1500 units of payoff, a salesperson does not necessarily need to receive \\$1500.)  What is the optimal allocation? What is the average payoff?

In [3]:
# Your solution for 1.3

#We will need to add a constraint here

import gurobipy as gp
from gurobipy import *
import numpy as np

A_1_3 = np.array([[1, 0.3, 0.2, 0, 0, 0, 0.2, 0, 0.4, 0],
              [0.2, 1, 0.2, 0, 0.1, 0, 0.2, 0.1, 0.3, 0],
              [0.2, 0.1, 1, 0.1, 0, 0, 0.4, 0.2, 0, 0],
              [0, 0.1, 0.2, 1, 0.3, 0, 0.1, 0.2, 0.3, 0.1],
              [0.1, 0.1, 0, 0.2, 1, 0.1, 0.2, 0, 0, 0.1],
              [0, 0, 0, 0.1, 0.1, 1, 0.1, 0, 0, 0],
              [0.2, 0.2, 0, 0.1, 0.3, 0, 1, 0.1, 0.2, 0.3],
              [0, 0, 0.3, 0, 0.1, 0, 0.3, 1, 0.1, 0.3],
              [0.4, 0.4, 0, 0.1, 0, 0, 0.1, 0.2, 1, 0.1],
              [0, 0, 0.1, 0.2, 0.1, 0, 0.3, 0.3, 0.2, 1]])
names_1_3=['S_1', 'S_2', 'S_3', 'S_4', 'S_5', 'S_6', 'S_7', 'S_8','S_9','S_10',]
People_1_3=len(A_1_3)



prob_1_3=gp.Model("Question 1.3")

#Create variable
X1_3=prob_1_3.addVars(People_1_3,lb=0,vtype=GRB.CONTINUOUS,name = [i for i in names_1_3])
#set objective function 
prob_1_3.setObjective(sum(A_1_3[i][j]*X1_3[i] for i in range(10) for j in range(10)),GRB.MAXIMIZE)

#set constraints
##nonnegativity
for i in range(People_1_3):
        prob_1_3.addConstr(X1_3[i]>=0)
    
prob_1_3.addConstr(sum(X1_3[i] for i in range(People_1_3))<=10000)
        
#this constraint will ensure that the payoff per person is above 1500
for i in range(People_1_3):
        prob_1_3.addConstr(sum(A_1_3[i][j]*X1_3[i] for j in range(10))>=1500)


#optimize problem
prob_1_3.optimize()    
    
#since the problem has no soulton currently evaluating an error    
print("The optimal allocation is as follows:")
for v in prob_1_3.getVars():
    print(v.varName, "=", round(v.x))
print("The average payoff is",round(prob_1_3.objVal)/10)


Gurobi Optimizer version 9.5.2 build v9.5.2rc0 (win64)
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 21 rows, 10 columns and 30 nonzeros
Model fingerprint: 0x4eb6a71f
Coefficient statistics:
  Matrix range     [1e+00, 2e+00]
  Objective range  [1e+00, 2e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+03, 1e+04]
Presolve removed 21 rows and 10 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.1021114e+04   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  2.102111367e+04
The optimal allocation is as follows:
S_1 = 714
S_2 = 714
S_3 = 750
S_4 = 652
S_5 = 833
S_6 = 1154
S_7 = 3134
S_8 = 714
S_9 = 652
S_10 = 682
The average payoff is 2102.1


### 1.4: (20 pts)
Based on the the previous scenario, Ms. Fraser is considering reducing the limit in (1.3) **for just one salesperson** from 1500 units to 1100 units while still maximizing the average payoff. That is, she wants to maximize the average payoff while one salesperson receives at least 1100, and the rest receive at least 1500 units of payoff.
Who is that salesperson? What would the average payoff be in this policy?

In [68]:
# Your solution for 1.4
#We will need to add a constraint here


import gurobipy as gp
from gurobipy import *
import numpy as np

A_1_4 = np.array([[1, 0.3, 0.2, 0, 0, 0, 0.2, 0, 0.4, 0],
              [0.2, 1, 0.2, 0, 0.1, 0, 0.2, 0.1, 0.3, 0],
              [0.2, 0.1, 1, 0.1, 0, 0, 0.4, 0.2, 0, 0],
              [0, 0.1, 0.2, 1, 0.3, 0, 0.1, 0.2, 0.3, 0.1],
              [0.1, 0.1, 0, 0.2, 1, 0.1, 0.2, 0, 0, 0.1],
              [0, 0, 0, 0.1, 0.1, 1, 0.1, 0, 0, 0],
              [0.2, 0.2, 0, 0.1, 0.3, 0, 1, 0.1, 0.2, 0.3],
              [0, 0, 0.3, 0, 0.1, 0, 0.3, 1, 0.1, 0.3],
              [0.4, 0.4, 0, 0.1, 0, 0, 0.1, 0.2, 1, 0.1],
              [0, 0, 0.1, 0.2, 0.1, 0, 0.3, 0.3, 0.2, 1]])
names_1_4=['S_1', 'S_2', 'S_3', 'S_4', 'S_5', 'S_6', 'S_7', 'S_8','S_9','S_10',]
People_1_4=len(A_1_4)



prob_1_4=gp.Model("Question 1.4")

#Create variable
X1_4=prob_1_4.addVars(People_1_4,lb=0,vtype=GRB.CONTINUOUS,name = [i for i in names_1_4])
#set objective function 
prob_1_4.setObjective(sum(A_1_4[i][j]*X1_4[j] for i in range(10) for j in range(10)),GRB.MAXIMIZE)

#set constraints
##nonnegativity
for i in range(People_1_4):
        prob_1_4.addConstr(X1_4[i]>=0)
    
prob_1_4.addConstr(sum(X1_4[i] for i in range(People_1_4))<=10000)
        
#this constraint will ensure that the payoff per person is above 1500
for i in range(People_1_4):
        prob_1_4.addConstr(sum(A_1_4[i][j]*X1_4[j] for i in range(10) for j in range(10))>=1500)


#optimize problem
prob_1_4.optimize()    
    
#since the problem has no soulton currently evaluating an error    
print("The optimal allocation is as follows:")
for v in prob_1_4.getVars():
    print(v.varName, "=", round(v.x))
print("The average payoff is",round(prob_1_4.objVal)/10)



Gurobi Optimizer version 9.5.2 build v9.5.2rc0 (win64)
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 21 rows, 10 columns and 120 nonzeros
Model fingerprint: 0x6e26ed5c
Coefficient statistics:
  Matrix range     [1e+00, 3e+00]
  Objective range  [1e+00, 3e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+03, 1e+04]
Presolve removed 19 rows and 2 columns
Presolve time: 0.01s
Presolved: 2 rows, 8 columns, 16 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.9000000e+04   0.000000e+00   0.000000e+00      0s
       0    2.9000000e+04   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  2.900000000e+04
The optimal allocation is as follows:
S_1 = 0
S_2 = 0
S_3 = 0
S_4 = 0
S_5 = 0
S_6 = 0
S_7 = 10000
S_8 = 0
S_9 = 0
S_10 = 0
The average payoff is 2900.0


## Question 2:

Mr. Lei informs Ms. Fraser that some of these salespeople are frenemies - friends and enemies. (They
are not enemies per se, but they envy each other.) So when one of them receives a prize, those who envy
them will feel **some negative payoff**. This means now the following matrix (called **B**) represents the payoff relation among them:

| | $S_1$ | $S_2$ | $S_3$ | $S_4$ | $S_5$ | $S_6$ | $S_7$ | $S_8$ | $S_9$ | $S_{10}$|
| -- | -- | -- | -- | -- | -- | -- | -- | -- | -- | -- |
|$S_1$|1 | 0.3 | -0.1 | 0 | 0  | -0.2  | 0  | -0.3 | 0.4 | 0.1 |
|$S_2$ | 0.2 | 1 | 0.1 | 0.1 | -0.1 | 0 | -0.1 | 0.1 | 0.3 | -0.2 |
|$S_3$| 0.2 | 0 | 1 | 0.1 | -0.1 | 0.2 | 0.2 | 0 | -0.3 | 0 |
|$S_4$| -0.1 | 0.2 | 0 | 1 | 0.2 | 0 | 0.2 | 0 | 0.1 | -0.1 |
|$S_5$| 0.1 | -0.1 | -0.1 | 0.1 | 1 | 0.1 | 0.1 | 0.3 | 0 | 0.1 | 
|$S_6$| -0.2 | -0.2 | -0.1 | 0.2 | 0.2 | 1 | 0.3 | -0.1 | 0 | -0.2 |
|$S_7$| 0.1 | 0.2 | -0.1 | -0.1 | 0.2 | -0.1 | 1 | 0.2 | 0.1 | 0.3 |
|$S_8$| 0.1 | 0.1 | 0.3 | -0.1 | 0.1 | 0.2 | 0.2 | 1 | 0.2 | 0.1 |
|$S_9$| 0.3 | 0.4 | -0.2 | 0.2 | 0.1 | 0 | 0.1 | -0.1 | 1 | -0.1 |
|$S_{10}$| -0.1 | -0.2 | 0.1 | 0.1 | 0 | -0.1 | 0.2 | 0.2 | 0.3 | 1 |

where $b_{i,j}$ (the element in the intersection of the i'th row and the j'th column) is the payoff salesperson $i$ feels when salesperson $j$ receives one dollar. The payoff is linear in the received award. This means: $$\text{payoff}_i = \sum_{j = 1}^{10} b_{i,j} x_j$$
where $x_j$ is the amount of money salesperson $j$ receives in the final allocation. Suppose, for instance, Ms. Fraser gives \\$2,000 to salesperson 1, \\$3,000 to salesperson 3, \\$3,000 to salesperson 5, and \\$2,000 to salesperson 7, the payoff of salesperson 3 would be:
$$\text{payoff}_3 = \sum_{j = 1}^{10} b_{3,j} x_j = 0.2 \times 2000 + 1 \times 3000 - 0.1 \times 3000 + 0.2 \times 2000 = 3500$$.

The following example will further help you understand in more details what this model means. Suppose Ms. Fraser gives \\$5,000 to salesperson 1, and \\$5,000 to salesperson 2. The following code will calculate the payoff of each salesperson resulting from this allocation:

In [8]:
B = np.array([[1, 0.3, -0.1, 0, 0, -0.2, 0, -0.3, 0.4, 0.1],
              [0.2, 1, 0.1, 0.1, -0.1, 0, -0.1, 0.1, 0.3, -0.2],
              [0.2, 0, 1, 0.1, -0.1, 0.2, 0.2, 0, -0.3, 0],
              [-0.1, 0.2, 0, 1, 0.2, 0, 0.2, 0, 0.1, -0.1],
              [0.1, -0.1, -0.1, 0.1, 1, 0.1, 0.1, 0.3, 0, 0.1], 
              [-0.2, -0.2, -0.1, 0.2, 0.2, 1, 0.3, -0.1, 0, -0.2],
              [0.1, 0.2, -0.1, -0.1, 0.2, -0.1, 1, 0.2, 0.1, 0.3],
              [0.1, 0.1, 0.3, -0.1, 0.1, 0.2, 0.2, 1, 0.2, 0.1],
              [0.3, 0.4, -0.2, 0.2, 0.1, 0, 0.1, -0.1, 1, -0.1],
              [-0.1, -0.2, 0.1, 0.1, 0, -0.1, 0.2, 0.2, 0.3, 1]])

allocation = np.array([5000, 5000, 0, 0, 0, 0, 0, 0, 0, 0])
payoff_B = np.empty(10)

# Let's calculate the payoff of salesperson i from this allocation:
for i in range(10):
    payoff_B[i] = sum(B[i][j] * allocation[j] for j in range(10))
    
for i in range(10):
    print("The payoff of salesperson", (i+1), "from this allocation is %d" % payoff_B[i])
    
# The reason for adding 1 to the index is that Python indices start from 0, while our salesperson indices start from 1.
# Therefore, for instance, payoff[6] correspondes to the payoff of salesperson 7, not 6.

The payoff of salesperson 1 from this allocation is 6500
The payoff of salesperson 2 from this allocation is 6000
The payoff of salesperson 3 from this allocation is 1000
The payoff of salesperson 4 from this allocation is 500
The payoff of salesperson 5 from this allocation is 0
The payoff of salesperson 6 from this allocation is -2000
The payoff of salesperson 7 from this allocation is 1500
The payoff of salesperson 8 from this allocation is 1000
The payoff of salesperson 9 from this allocation is 3500
The payoff of salesperson 10 from this allocation is -1500


And the **average payoff** (the sum of the payoff of all salespersons divided by their number) with this allocation would be:

In [9]:
average_payoff = sum(payoff_B[i] for i in range(10)) / 10
print("The average payoff of this allocation is %d." % average_payoff)

The average payoff of this allocation is 1650.


### Now that you understood what is going on here, consider the following cases and answer the questions using matrix **B**:

## 2.1: (5 pts)
Suppose Ms. Fraser deeply believes in equality, and her interpretation of equality is equally distributing the award among all ten salespeople - i.e., she allocates \\$1,000 to each. What is the average payoff of this policy?

In [71]:
# Your solution for 2.1
B = np.array([[1, 0.3, -0.1, 0, 0, -0.2, 0, -0.3, 0.4, 0.1],
              [0.2, 1, 0.1, 0.1, -0.1, 0, -0.1, 0.1, 0.3, -0.2],
              [0.2, 0, 1, 0.1, -0.1, 0.2, 0.2, 0, -0.3, 0],
              [-0.1, 0.2, 0, 1, 0.2, 0, 0.2, 0, 0.1, -0.1],
              [0.1, -0.1, -0.1, 0.1, 1, 0.1, 0.1, 0.3, 0, 0.1], 
              [-0.2, -0.2, -0.1, 0.2, 0.2, 1, 0.3, -0.1, 0, -0.2],
              [0.1, 0.2, -0.1, -0.1, 0.2, -0.1, 1, 0.2, 0.1, 0.3],
              [0.1, 0.1, 0.3, -0.1, 0.1, 0.2, 0.2, 1, 0.2, 0.1],
              [0.3, 0.4, -0.2, 0.2, 0.1, 0, 0.1, -0.1, 1, -0.1],
              [-0.1, -0.2, 0.1, 0.1, 0, -0.1, 0.2, 0.2, 0.3, 1]])

allocation = np.array([1000, 1000, 1000, 1000, 1000, 1000,1000 , 1000, 1000, 1000])
payoff_B = np.empty(10)

# Let's calculate the payoff of salesperson i from this allocation:
for i in range(10):
    payoff_B[i] = sum(B[i][j] * allocation[j] for j in range(10))
    
for i in range(10):
    print("The payoff of salesperson", (i+1), "from this allocation is %d" % payoff_B[i])

average_payoff = sum(payoff_B[i] for i in range(10)) / 10
print("The average payoff of this allocation is %d." % average_payoff)  


The payoff of salesperson 1 from this allocation is 1200
The payoff of salesperson 2 from this allocation is 1400
The payoff of salesperson 3 from this allocation is 1300
The payoff of salesperson 4 from this allocation is 1500
The payoff of salesperson 5 from this allocation is 1600
The payoff of salesperson 6 from this allocation is 900
The payoff of salesperson 7 from this allocation is 1800
The payoff of salesperson 8 from this allocation is 2200
The payoff of salesperson 9 from this allocation is 1700
The payoff of salesperson 10 from this allocation is 1500
The average payoff of this allocation is 1510.


## 2.2: (10 pts)
Suppose Ms. Fraser now wants to maximize the average payoff. What is the optimal award allocation? What is the average payoff? (Hint: you can solve this without help from Gurobi or any optimization program. But if you are not 100% sure of your solution, I suggest playing safe and using Gurobi.)

In [74]:
# Your solution for 2.2
import gurobipy as gp
from gurobipy import *
import numpy as np

B_2_2 = np.array([[1, 0.3, -0.1, 0, 0, -0.2, 0, -0.3, 0.4, 0.1],
              [0.2, 1, 0.1, 0.1, -0.1, 0, -0.1, 0.1, 0.3, -0.2],
              [0.2, 0, 1, 0.1, -0.1, 0.2, 0.2, 0, -0.3, 0],
              [-0.1, 0.2, 0, 1, 0.2, 0, 0.2, 0, 0.1, -0.1],
              [0.1, -0.1, -0.1, 0.1, 1, 0.1, 0.1, 0.3, 0, 0.1], 
              [-0.2, -0.2, -0.1, 0.2, 0.2, 1, 0.3, -0.1, 0, -0.2],
              [0.1, 0.2, -0.1, -0.1, 0.2, -0.1, 1, 0.2, 0.1, 0.3],
              [0.1, 0.1, 0.3, -0.1, 0.1, 0.2, 0.2, 1, 0.2, 0.1],
              [0.3, 0.4, -0.2, 0.2, 0.1, 0, 0.1, -0.1, 1, -0.1],
              [-0.1, -0.2, 0.1, 0.1, 0, -0.1, 0.2, 0.2, 0.3, 1]])

names=['S_1', 'S_2', 'S_3', 'S_4', 'S_5', 'S_6', 'S_7', 'S_8','S_9','S_10',]
People=len(names)

prob_2_2=gp.Model("Question 2.2")

#Create variable
X2_2=prob_2_2.addVars(People,lb=0,vtype=GRB.CONTINUOUS,name = [i for i in names])
#set objective function 
prob_2_2.setObjective(sum(B_2_2[i][j]*X2_2[j] for i in range(10) for j in range(10)),GRB.MINIMIZE)

#set constraints
##nonnegativity
for i in range(People):
        prob_2_2.addConstr(X2_2[i]>=0)

#force gurobi to use all budget
prob_2_2.addConstr(sum(X2_2[i] for i in range(People))==10000)
        


#optimize problem
prob_2_2.optimize()    
    
#since the problem has no soulton currently evaluating an error    
print("The optimal allocation is as follows:")
for v in prob_2_2.getVars():
    print(v.varName, "=", round(v.x))
print("The average payoff is",round(prob_2_2.objVal)/10)


Gurobi Optimizer version 9.5.2 build v9.5.2rc0 (win64)
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 11 rows, 10 columns and 20 nonzeros
Model fingerprint: 0x9a465e60
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [9e-01, 2e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+04, 1e+04]
Presolve removed 11 rows and 10 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    9.0000000e+03   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  9.000000000e+03
The optimal allocation is as follows:
S_1 = 0
S_2 = 0
S_3 = 10000
S_4 = 0
S_5 = 0
S_6 = 0
S_7 = 0
S_8 = 0
S_9 = 0
S_10 = 0
The average payoff is 900.0


## 2.3: (20 pts)
Ms. Fraser is curious about the **worst way of distributing the award** among the salespeople. So she wants to find the allocation that **minimizes the average payoff while distributing all the \\$10,000 award**. What is the worst award allocation? What is the average payoff? (Hint: like the previous scenario, you can solve this without help from Gurobi or any optimization program. But if you are not 100% sure of your solution, I suggest playing safe and using Gurobi.)

In [12]:
# Your solution for 2.3

## Question 3:

Now that Mr. Lei has Ms. Fraser’s attention, he explains some of the difficulties and nuances
of the tasks they deal with in HR, probably hoping to get some additional budget for his department. He
tells the manager that these payoff matrices cannot be very precise by nature, and, especially when working on
human-related issues, there is always some debate about the methods. That is the reason behind
having two different payoff matrices. Nevertheless, Mr. Lei says the first matrix (**A**) comes from a less reliable
method than the second matrix (**B**). So with $p = 1/3$ the true matrix is the first one and with $p = 2/3$ the true matrix is the second one.


The following example will further help you understand how this uncertainty works. Suppose Ms. Fraser gives \\$1,000 to each salesperson. The following code will calculate the **expected payoff** of each salesperson resulting from this allocation:

In [13]:
A = np.array([[1, 0.3, 0.2, 0, 0, 0, 0.2, 0, 0.4, 0],
              [0.2, 1, 0.2, 0, 0.1, 0, 0.2, 0.1, 0.3, 0],
              [0.2, 0.1, 1, 0.1, 0, 0, 0.4, 0.2, 0, 0],
              [0, 0.1, 0.2, 1, 0.3, 0, 0.1, 0.2, 0.3, 0.1],
              [0.1, 0.1, 0, 0.2, 1, 0.1, 0.2, 0, 0, 0.1],
              [0, 0, 0, 0.1, 0.1, 1, 0.1, 0, 0, 0],
              [0.2, 0.2, 0, 0.1, 0.3, 0, 1, 0.1, 0.2, 0.3],
              [0, 0, 0.3, 0, 0.1, 0, 0.3, 1, 0.1, 0.3],
              [0.4, 0.4, 0, 0.1, 0, 0, 0.1, 0.2, 1, 0.1],
              [0, 0, 0.1, 0.2, 0.1, 0, 0.3, 0.3, 0.2, 1]])

B = np.array([[1, 0.3, -0.1, 0, 0, -0.2, 0, -0.3, 0.4, 0.1],
              [0.2, 1, 0.1, 0.1, -0.1, 0, -0.1, 0.1, 0.3, -0.2],
              [0.2, 0, 1, 0.1, -0.1, 0.2, 0.2, 0, -0.3, 0],
              [-0.1, 0.2, 0, 1, 0.2, 0, 0.2, 0, 0.1, -0.1],
              [0.1, -0.1, -0.1, 0.1, 1, 0.1, 0.1, 0.3, 0, 0.1], 
              [-0.2, -0.2, -0.1, 0.2, 0.2, 1, 0.3, -0.1, 0, -0.2],
              [0.1, 0.2, -0.1, -0.1, 0.2, -0.1, 1, 0.2, 0.1, 0.3],
              [0.1, 0.1, 0.3, -0.1, 0.1, 0.2, 0.2, 1, 0.2, 0.1],
              [0.3, 0.4, -0.2, 0.2, 0.1, 0, 0.1, -0.1, 1, -0.1],
              [-0.1, -0.2, 0.1, 0.1, 0, -0.1, 0.2, 0.2, 0.3, 1]])

allocation = np.array([1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000])
expected_payoff = np.empty(10)

# Let's calculate the expected payoff of salesperson i from this allocation:
for i in range(10):
    expected_payoff[i] = (1/3) * sum(A[i][j] * allocation[j] for j in range(10)) + (2/3) * sum(B[i][j] * allocation[j] for j in range(10))

for i in range(10):
    print("The expected payoff of salesperson", (i+1), "from this allocation is %d" % expected_payoff[i])
    
# The reason for adding 1 to the index is that Python indices start from 0, while our salesperson indices start from 1.
# Therefore, for instance, payoff[6] correspondes to the payoff of salesperson 7, not 6.

The expected payoff of salesperson 1 from this allocation is 1500
The expected payoff of salesperson 2 from this allocation is 1633
The expected payoff of salesperson 3 from this allocation is 1533
The expected payoff of salesperson 4 from this allocation is 1766
The expected payoff of salesperson 5 from this allocation is 1666
The expected payoff of salesperson 6 from this allocation is 1033
The expected payoff of salesperson 7 from this allocation is 2000
The expected payoff of salesperson 8 from this allocation is 2166
The expected payoff of salesperson 9 from this allocation is 1900
The expected payoff of salesperson 10 from this allocation is 1733


In [14]:
expected_average_payoff = sum(expected_payoff[i] for i in range(10)) / 10
print("The expected average payoff of this allocation is %d." % expected_average_payoff)

The expected average payoff of this allocation is 1693.


## 3.1: (10 pts)
Suppose Ms. Fraser wants to ensure that **the expected payoff of each salesperson is at least 1000 units.**
What is the optimal allocation? What is the expected average payoff?

In [79]:
# Your solution for 3.1

import gurobipy as gp
from gurobipy import *
import numpy as np
A = np.array([[1, 0.3, 0.2, 0, 0, 0, 0.2, 0, 0.4, 0],
              [0.2, 1, 0.2, 0, 0.1, 0, 0.2, 0.1, 0.3, 0],
              [0.2, 0.1, 1, 0.1, 0, 0, 0.4, 0.2, 0, 0],
              [0, 0.1, 0.2, 1, 0.3, 0, 0.1, 0.2, 0.3, 0.1],
              [0.1, 0.1, 0, 0.2, 1, 0.1, 0.2, 0, 0, 0.1],
              [0, 0, 0, 0.1, 0.1, 1, 0.1, 0, 0, 0],
              [0.2, 0.2, 0, 0.1, 0.3, 0, 1, 0.1, 0.2, 0.3],
              [0, 0, 0.3, 0, 0.1, 0, 0.3, 1, 0.1, 0.3],
              [0.4, 0.4, 0, 0.1, 0, 0, 0.1, 0.2, 1, 0.1],
              [0, 0, 0.1, 0.2, 0.1, 0, 0.3, 0.3, 0.2, 1]])

B = np.array([[1, 0.3, -0.1, 0, 0, -0.2, 0, -0.3, 0.4, 0.1],
              [0.2, 1, 0.1, 0.1, -0.1, 0, -0.1, 0.1, 0.3, -0.2],
              [0.2, 0, 1, 0.1, -0.1, 0.2, 0.2, 0, -0.3, 0],
              [-0.1, 0.2, 0, 1, 0.2, 0, 0.2, 0, 0.1, -0.1],
              [0.1, -0.1, -0.1, 0.1, 1, 0.1, 0.1, 0.3, 0, 0.1], 
              [-0.2, -0.2, -0.1, 0.2, 0.2, 1, 0.3, -0.1, 0, -0.2],
              [0.1, 0.2, -0.1, -0.1, 0.2, -0.1, 1, 0.2, 0.1, 0.3],
              [0.1, 0.1, 0.3, -0.1, 0.1, 0.2, 0.2, 1, 0.2, 0.1],
              [0.3, 0.4, -0.2, 0.2, 0.1, 0, 0.1, -0.1, 1, -0.1],
              [-0.1, -0.2, 0.1, 0.1, 0, -0.1, 0.2, 0.2, 0.3, 1]])

names=['S_1', 'S_2', 'S_3', 'S_4', 'S_5', 'S_6', 'S_7', 'S_8','S_9','S_10',]
People=len(names)

prob_3_1=gp.Model("Question 3.1")

#Create variable
X=prob_3_1.addVars(People,lb=0,vtype=GRB.CONTINUOUS,name = [i for i in names])
#set objective function 
prob_3_1.setObjective(sum((1/3*A[i][j]+2/3*B[i][j])*X[j] for i in range(10) for j in range(10)),GRB.MAXIMIZE)

#set constraints
##nonnegativity
for i in range(People):
        prob_3_1.addConstr(X[i]>=0)

for i in range(People):
    prob_3_1.addConstr(sum((1/3*A[i][j]+2/3*B[i][j])*X[j] for j in range(10))>=1000)

#optimize problem
prob_3_1.optimize()    
    
#since the problem has no soulton currently evaluating an error    
print("The optimal allocation is as follows:")
for v in prob_3_1.getVars():
    print(v.varName, "=", round(v.x))
print("The average payoff is",round(prob_3_1.objVal)/10)

Gurobi Optimizer version 9.5.2 build v9.5.2rc0 (win64)
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 20 rows, 10 columns and 99 nonzeros
Model fingerprint: 0xf7a1f042
Coefficient statistics:
  Matrix range     [3e-02, 1e+00]
  Objective range  [1e+00, 2e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+03, 1e+03]
Presolve time: 0.01s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Infeasible or unbounded model
The optimal allocation is as follows:


AttributeError: Unable to retrieve attribute 'x'

## 3.2: (20 pts)
Suppose Ms. Fraser wants to ensure each salesperson receives at least 1000 units of payoff **regardless
of which matrix is the correct one**, while maximizing the expected average payoff. Solve it, and
explain how it differs from (3.1), where Ms. Fraser wanted each of them to receive at least 1000 units of **expected
payoff.**

In [16]:
# Your solution for 3.2
A = np.array([[1, 0.3, 0.2, 0, 0, 0, 0.2, 0, 0.4, 0],
              [0.2, 1, 0.2, 0, 0.1, 0, 0.2, 0.1, 0.3, 0],
              [0.2, 0.1, 1, 0.1, 0, 0, 0.4, 0.2, 0, 0],
              [0, 0.1, 0.2, 1, 0.3, 0, 0.1, 0.2, 0.3, 0.1],
              [0.1, 0.1, 0, 0.2, 1, 0.1, 0.2, 0, 0, 0.1],
              [0, 0, 0, 0.1, 0.1, 1, 0.1, 0, 0, 0],
              [0.2, 0.2, 0, 0.1, 0.3, 0, 1, 0.1, 0.2, 0.3],
              [0, 0, 0.3, 0, 0.1, 0, 0.3, 1, 0.1, 0.3],
              [0.4, 0.4, 0, 0.1, 0, 0, 0.1, 0.2, 1, 0.1],
              [0, 0, 0.1, 0.2, 0.1, 0, 0.3, 0.3, 0.2, 1]])

B = np.array([[1, 0.3, -0.1, 0, 0, -0.2, 0, -0.3, 0.4, 0.1],
              [0.2, 1, 0.1, 0.1, -0.1, 0, -0.1, 0.1, 0.3, -0.2],
              [0.2, 0, 1, 0.1, -0.1, 0.2, 0.2, 0, -0.3, 0],
              [-0.1, 0.2, 0, 1, 0.2, 0, 0.2, 0, 0.1, -0.1],
              [0.1, -0.1, -0.1, 0.1, 1, 0.1, 0.1, 0.3, 0, 0.1], 
              [-0.2, -0.2, -0.1, 0.2, 0.2, 1, 0.3, -0.1, 0, -0.2],
              [0.1, 0.2, -0.1, -0.1, 0.2, -0.1, 1, 0.2, 0.1, 0.3],
              [0.1, 0.1, 0.3, -0.1, 0.1, 0.2, 0.2, 1, 0.2, 0.1],
              [0.3, 0.4, -0.2, 0.2, 0.1, 0, 0.1, -0.1, 1, -0.1],
              [-0.1, -0.2, 0.1, 0.1, 0, -0.1, 0.2, 0.2, 0.3, 1]])

names=['S_1', 'S_2', 'S_3', 'S_4', 'S_5', 'S_6', 'S_7', 'S_8','S_9','S_10',]
People=len(names)

prob_3_2=gp.Model("Question 3.1")

#Create variable
X=prob_3_2.addVars(People,lb=0,vtype=GRB.CONTINUOUS,name = [i for i in names])
#set objective function 
prob_3_2.setObjective(sum((A[i][j]+B[i][j])*X[j] for i in range(10) for j in range(10)),GRB.MAXIMIZE)

#set constraints
##nonnegativity
for i in range(People):
        prob_3_2.addConstr(X[i]>=0)

for i in range(People):
    prob_3_2.addConstr(sum((1/3*A[i][j]+2/3*B[i][j])*X[j] for j in range(10))>=1000)

#optimize problem
prob_3_2.optimize()    
    
#since the problem has no soulton currently evaluating an error    
print("The optimal allocation is as follows:")
for v in prob_3_2.getVars():
    print(v.varName, "=", round(v.x))
print("The average payoff is",round(prob_3_2.objVal)/10)


The key difference here is that we are not using the expected value to determine what the payoff is. The proabilities are now unknown and therefore have to be treated with the same weight. The problem is therefore looking to solve how we can maximize the payoff if both are correct therefore giving them equal weight. 